# LAB 08 — Tokenization & Attention Mask

Tokenizer: `distilbert-base-uncased` (WordPiece, cùng tokenizer với model dùng trong `fine_tuning.ipynb`).

Nội dung: Calculation Exercise 2 (section 10) và Core Implementation 12.1 — chạy tokenizer, ghi token sequence, token IDs,
special tokens, attention mask, so sánh số words / tokens / subword tokens.

In [1]:
# Setup: on Google Colab (if the code is not already here), clone the repo and move into lab08/
import os, sys
if "google.colab" in sys.modules and not os.path.exists("pretrained_classification.py"):
    if not os.path.exists("/content/NLPandApp"):
        !git clone -q https://github.com/DuanShine/NLPandApp.git /content/NLPandApp
    %cd /content/NLPandApp/lab08
    !pip -q install -U datasets
print(os.getcwd())

/content/lab08


In [2]:
import re
import pandas as pd
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(type(tokenizer).__name__, "| vocab size:", tokenizer.vocab_size)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BertTokenizer | vocab size: 30522


## 1. Tokenize câu `"The cats are playing."`

In [3]:
text = "The cats are playing."
tokens = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
print("keys:", list(tokens.keys()))
print("input_ids     :", tokens["input_ids"].tolist())
print("attention_mask:", tokens["attention_mask"].tolist())

keys: ['input_ids', 'token_type_ids', 'attention_mask']
input_ids     : [[101, 1996, 8870, 2024, 2652, 1012, 102]]
attention_mask: [[1, 1, 1, 1, 1, 1, 1]]


## 2–5. Token sequence, token IDs, special tokens, attention mask

In [4]:
ids = tokens["input_ids"][0].tolist()
toks = tokenizer.convert_ids_to_tokens(ids)
special_ids = set(tokenizer.all_special_ids)
table = pd.DataFrame({
    "position": range(len(ids)),
    "token": toks,
    "token_id": ids,
    "special?": [i in special_ids for i in ids],
    "subword piece (##)?": [t.startswith("##") for t in toks],
    "attention_mask": tokens["attention_mask"][0].tolist(),
})
table

,position,token,token_id,special?,subword piece (##)?,attention_mask
0,0,[CLS],101,True,False,1
1,1,the,1996,False,False,1
2,2,cats,8870,False,False,1
3,3,are,2024,False,False,1
4,4,playing,2652,False,False,1
5,5,.,1012,False,False,1
6,6,[SEP],102,True,False,1


In [5]:
print("Special tokens of this tokenizer:")
for name, tok in tokenizer.special_tokens_map.items():
    print(f"  {name:10s} {tok:8s} id={tokenizer.convert_tokens_to_ids(tok)}")
print("Special tokens in this sentence:", [t for t, i in zip(toks, ids) if i in special_ids])

Special tokens of this tokenizer:
  unk_token  [UNK]    id=100
  sep_token  [SEP]    id=102
  pad_token  [PAD]    id=0
  cls_token  [CLS]    id=101
  mask_token [MASK]   id=103
Special tokens in this sentence: ['[CLS]', '[SEP]']


## 6. So sánh số lượng words / tokens / subword tokens

In [6]:
def count_report(text):
    words = re.findall(r"\w+|[^\w\s]", text)          # words + punctuation
    alpha_words = re.findall(r"\w+", text)              # words only
    enc = tokenizer(text)
    toks = tokenizer.convert_ids_to_tokens(enc["input_ids"])
    content = [t for t in toks if t not in tokenizer.all_special_tokens]
    per_word = {w: tokenizer.tokenize(w) for w in alpha_words}
    return {
        "text": text,
        "words (no punct)": len(alpha_words),
        "tokens incl. special": len(toks),
        "tokens excl. special": len(content),
        "'##' continuation pieces": sum(t.startswith("##") for t in content),
        "words split into >1 piece": sum(len(p) > 1 for p in per_word.values()),
        "per-word split": per_word,
    }

pd.DataFrame([count_report(t) for t in [
    "The cats are playing.",
    "The cat eats fish.",
    "This movie was unbelievably overhyped and underwhelming.",
    "Tokenization handles unseen words like Transformerization.",
]])

,text,words (no punct),tokens incl. special,tokens excl. special,'##' continuation pieces,words split into >1 piece,per-word split
0,The cats are playing.,4,7,5,0,0,"{'The': ['the'], 'cats': ['cats'], 'are': ['ar..."
1,The cat eats fish.,4,7,5,0,0,"{'The': ['the'], 'cat': ['cat'], 'eats': ['eat..."
2,This movie was unbelievably overhyped and unde...,7,19,17,9,3,"{'This': ['this'], 'movie': ['movie'], 'was': ..."
3,Tokenization handles unseen words like Transfo...,6,12,10,3,2,"{'Tokenization': ['token', '##ization'], 'hand..."


## 7. Padding và attention mask trong một batch

Khi nhiều câu có độ dài khác nhau được đưa vào cùng batch, câu ngắn được thêm `[PAD]`; attention mask = 0 ở các vị trí đó.

In [7]:
batch = ["The cats are playing.", "The cat eats fish", "Cats play."]
enc = tokenizer(batch, padding=True, truncation=True, return_tensors="pt")
for i, s in enumerate(batch):
    print(f"{s!r}")
    print("   tokens:", tokenizer.convert_ids_to_tokens(enc["input_ids"][i]))
    print("   ids   :", enc["input_ids"][i].tolist())
    print("   mask  :", enc["attention_mask"][i].tolist())

'The cats are playing.'
   tokens: ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]']
   ids   : [101, 1996, 8870, 2024, 2652, 1012, 102]
   mask  : [1, 1, 1, 1, 1, 1, 1]
'The cat eats fish'
   tokens: ['[CLS]', 'the', 'cat', 'eats', 'fish', '[SEP]', '[PAD]']
   ids   : [101, 1996, 4937, 20323, 3869, 102, 0]
   mask  : [1, 1, 1, 1, 1, 1, 0]
'Cats play.'
   tokens: ['[CLS]', 'cats', 'play', '.', '[SEP]', '[PAD]', '[PAD]']
   ids   : [101, 8870, 2377, 1012, 102, 0, 0]
   mask  : [1, 1, 1, 1, 1, 0, 0]


## 8. Truncation (max_length)

In [8]:
long_text = "great " * 300
for L in (64, 128, 256):
    e = tokenizer(long_text, truncation=True, max_length=L)
    print(L, "->", len(e["input_ids"]), "tokens; last token:", tokenizer.convert_ids_to_tokens(e["input_ids"][-1]))

64 -> 64 tokens; last token: [SEP]
128 -> 128 tokens; last token: [SEP]
256 -> 256 tokens; last token: [SEP]


## Nhận xét của sinh viên

- `input_ids`: _(tự viết)_
- `attention_mask`: _(tự viết)_
- Special tokens (`[CLS]`, `[SEP]`, `[PAD]`): _(tự viết)_
- Words vs tokens vs subword tokens: _(tự viết)_